# 01 — Espacio de estados y búsqueda para sueño infantil
## Caso Tipo C: Sueño en los Recién Nacidos

**Objetivo:** crear estados sintéticos de cinco familias, calcular su costo de desviación y seleccionar el estado de menor costo. Los datos y la fórmula son ficticios y se usan solo para aprender programación.

## 1. Importar bibliotecas y definir la función de costo

Se usa la misma función de costo en ambos notebooks. El componente de sueño se limita a cero para evitar costos negativos si el sueño continuo supera 10 horas.

In [ ]:
import heapq
import random
from dataclasses import dataclass

def calcular_costo(horas_sueno, despertarse, fatiga_padres, rutina):
    componente_sueno = max(0, (10 - horas_sueno) * 15)
    return (
        componente_sueno
        + despertarse * 20
        + fatiga_padres * 15
        + (2 - rutina) * 10
    )

print("Función de costo definida.")

## 2. Representar el espacio de estados

Cada estado contiene los valores sintéticos de una familia. La rutina toma valores entre 0 y 2 para que la fórmula tenga una interpretación consistente dentro de este ejemplo.

In [ ]:
@dataclass
class EstadoSuenoInfantil:
    id_familia: str
    horas_sueno: float
    despertarse: int
    fatiga_padres: int
    rutina: int

    @property
    def costo_total(self):
        return calcular_costo(
            self.horas_sueno,
            self.despertarse,
            self.fatiga_padres,
            self.rutina
        )

    def __lt__(self, other):
        return self.costo_total < other.costo_total

## 3. Generar estados iniciales S0

La semilla aleatoria permite reproducir los mismos ejemplos cada vez que se ejecuta el notebook.

In [ ]:
random.seed(42)
familias = ["Familia A", "Familia B", "Familia C", "Familia D", "Familia E"]

espacio_estados = [
    EstadoSuenoInfantil(
        id_familia=f,
        horas_sueno=random.uniform(2.5, 10.5),
        despertarse=random.randint(0, 5),
        fatiga_padres=random.randint(0, 4),
        rutina=random.randint(0, 2)
    )
    for f in familias
]

print("ESPACIO DE ESTADOS INICIALES")
for e in espacio_estados:
    print(
        f"{e.id_familia} -> Sueño: {e.horas_sueno:.1f} h | "
        f"Despertares: {e.despertarse} | Fatiga: {e.fatiga_padres} | "
        f"Rutina: {e.rutina} | Costo: {e.costo_total:.2f}"
    )

## 4. Búsqueda de menor costo

Como cada familia es un estado candidato independiente y no se definieron transiciones entre familias, este bloque usa una cola de prioridad para ordenar candidatos por costo. Es una búsqueda de mejor candidato, **no una demostración completa de A*** en un grafo.

In [ ]:
def buscar_estado_menor_costo(espacio_estados, costo_objetivo=None):
    frontera = [(e.costo_total, e) for e in espacio_estados]
    heapq.heapify(frontera)

    mejor = None
    while frontera:
        costo, actual = heapq.heappop(frontera)
        if mejor is None:
            mejor = actual
        if costo_objetivo is not None and costo <= costo_objetivo:
            return actual
    return mejor

costo_meta = 55.0
optimo = buscar_estado_menor_costo(espacio_estados, costo_objetivo=costo_meta)

print("--- ESTADO SELECCIONADO ---")
print(f"Familia: {optimo.id_familia}")
print(f"Sueño continuo: {optimo.horas_sueno:.1f} h")
print(f"Despertares: {optimo.despertarse}")
print(f"Fatiga de padres: {optimo.fatiga_padres}")
print(f"Rutina (escala ficticia): {optimo.rutina}")
print(f"Costo de desviación: {optimo.costo_total:.2f}")
if optimo.costo_total > costo_meta:
    print(f"No se encontró un candidato con costo <= {costo_meta:.1f}; se muestra el de menor costo disponible.")
else:
    print(f"Se encontró un candidato que cumple costo <= {costo_meta:.1f}.")

## 5. Visualizar costos por estado

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_familias = pd.DataFrame([{
    "Familia": e.id_familia,
    "X1_Sueno": e.horas_sueno,
    "X2_Despertarse": e.despertarse,
    "X3_Fatiga": e.fatiga_padres,
    "X4_Rutina": e.rutina,
    "Costo_Total": e.costo_total
} for e in espacio_estados])

display(df_familias.round(2))

plt.figure(figsize=(9, 4))
plt.bar(df_familias["Familia"], df_familias["Costo_Total"])
plt.axhline(costo_meta, linestyle="--", label="Costo objetivo")
plt.title("Costo sintético por estado/familia")
plt.xlabel("Estado")
plt.ylabel("Costo de desviación (ficticio)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Conclusión
Se definió un espacio de estados, se calculó una función de costo y se seleccionó el estado candidato con menor costo. Los valores y pesos son inventados para el ejercicio; no describen estándares médicos ni permiten recomendar rutinas reales.